# 📐 Clase 4 — Regresión lineal múltiple: fundamentos y estructura del modelo
## Unidad: Correlación y modelamiento

**Situación:** Una plataforma de educación digital quiere explicar qué factores influyen en el rendimiento final de sus estudiantes. Las variables disponibles son: tiempo de estudio, interacciones en foros, sesiones activas y notas previas.

**Preguntas clave:**
- ¿Podemos combinar todas las variables para explicar el rendimiento?
- ¿Qué supuestos deben cumplirse para confiar en el modelo?
- ¿Qué variables conviene conservar?

**Objetivos:**
- Construir un modelo **Y = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ + ε**
- Verificar los **5 supuestos** del modelo lineal clásico
- Detectar **multicolinealidad** con VIF
- Aplicar **selección de variables** (backward/AIC)
- Comparar modelo completo vs modelo reducido

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')

---
## PARTE 1 — De simple a múltiple: ¿por qué necesitamos más variables?

### 1.1 Limitación del modelo simple

In [ ]:
np.random.seed(42)
n_demo = 80

tiempo   = np.random.normal(10, 2, n_demo)
productos = np.random.randint(2, 10, n_demo).astype(float)
gasto     = 500 + 180 * tiempo + 120 * productos + np.random.normal(0, 300, n_demo)

df_demo = pd.DataFrame({'tiempo': tiempo, 'productos_vistos': productos, 'gasto': gasto})

# Modelo simple vs múltiple
X_simple   = sm.add_constant(df_demo[['tiempo']])
X_multiple = sm.add_constant(df_demo[['tiempo', 'productos_vistos']])
Y_demo     = df_demo['gasto']

m_simple   = sm.OLS(Y_demo, X_simple).fit()
m_multiple = sm.OLS(Y_demo, X_multiple).fit()

print('=== Modelo SIMPLE (solo tiempo) ===')
print(f'  R²: {m_simple.rsquared:.4f}  |  AIC: {m_simple.aic:.2f}')
print()
print('=== Modelo MÚLTIPLE (tiempo + productos_vistos) ===')
print(f'  R²: {m_multiple.rsquared:.4f}  |  AIC: {m_multiple.aic:.2f}')
print()
print(f'Ganancia de R²: +{(m_multiple.rsquared - m_simple.rsquared)*100:.1f}pp al añadir productos_vistos')

### 1.2 Ejemplo introductorio de la presentación

In [ ]:
# Datos exactos de la presentación
data = {
    'tiempo':           [5, 6, 7, 8, 5.5, 6.5],
    'productos_vistos': [3, 4, 5, 6, 3,   5],
    'gasto':            [1200, 1400, 1600, 1800, 1300, 1500]
}
df_intro = pd.DataFrame(data)

X_i = df_intro[['tiempo', 'productos_vistos']]
X_i = sm.add_constant(X_i)
Y_i = df_intro['gasto']
modelo_intro = sm.OLS(Y_i, X_i).fit()
print(modelo_intro.summary())

In [ ]:
b = modelo_intro.params
print('=== Interpretación de coeficientes (ceteris paribus) ===')
print(f'β₀ = {b["const"]:,.0f}  → gasto base cuando tiempo=0 y productos=0')
print(f'β₁ = {b["tiempo"]:,.0f}  → por cada minuto adicional de navegación,')
print(f'          el gasto aumenta ${b["tiempo"]:,.0f}, si productos_vistos se mantiene constante')
print(f'β₂ = {b["productos_vistos"]:,.0f}  → por cada producto adicional visto,')
print(f'          el gasto aumenta ${b["productos_vistos"]:,.0f}, si tiempo se mantiene constante')
print()
print('Ecuación: ŷ = ' +
      f'{b["const"]:,.0f} + {b["tiempo"]:,.0f}·tiempo + {b["productos_vistos"]:,.0f}·productos_vistos')

### ✏️ Ejercicio 1:

In [ ]:
# ✏️ ¿Qué significa "manteniendo las demás constantes" al interpretar β₁?
r_ceteris = ""

# ✏️ ¿Por qué puede ser mejor un modelo múltiple que uno simple, incluso con mayor complejidad?
r_ventaja = ""

# ✏️ Predice el gasto de un cliente que navega 7 min y ve 4 productos:
pred = b['const'] + b['tiempo']*7 + b['productos_vistos']*4
print(f'Predicción (7 min, 4 productos): ${pred:,.0f}')

print(f'Ceteris paribus: {r_ceteris}')
print(f'Ventaja múltiple: {r_ventaja}')

---
## PARTE 2 — Actividad guiada: Plataforma educativa

### 2.1 Generar el dataset exacto de la presentación

In [ ]:
# Código exacto de la presentación
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 100
df = pd.DataFrame({
    'tiempo_estudio':     np.random.normal(10, 2, n),
    'interacciones_foro': np.random.poisson(5, n),
    'sesiones_activas':   np.random.randint(3, 10, n),
    'notas_previas':      np.random.normal(4.5, 0.5, n),
})
# Variable dependiente con algo de ruido
df['rendimiento_final'] = (1.5 * df['tiempo_estudio']
                           + 0.8 * df['notas_previas']
                           + 0.3 * df['interacciones_foro']
                           + np.random.normal(0, 2, n))

print(df.describe().round(3))

### 2.2 Exploración visual previa (siempre antes de modelar)

In [ ]:
# Matriz de correlaciones
corr = df.corr().round(3)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title('Matriz de correlación — Plataforma educativa', fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlaciones con rendimiento_final:')
print(corr['rendimiento_final'].sort_values(ascending=False))

### 2.3 Ajustar el modelo completo (Paso exacto de la presentación)

In [ ]:
# Código exacto de la presentación
X = df[['tiempo_estudio', 'interacciones_foro', 'sesiones_activas', 'notas_previas']]
X = sm.add_constant(X)
Y = df['rendimiento_final']
modelo = sm.OLS(Y, X).fit()
print(modelo.summary())

In [ ]:
print('=== Resumen de coeficientes y significancia ===')
tabla_coef = pd.DataFrame({
    'coef':    modelo.params.round(4),
    'p-value': modelo.pvalues.round(4),
    'sig':     modelo.pvalues.apply(lambda p:
                   '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '(ns)')
})
print(tabla_coef)
print()
print(f'R² total:      {modelo.rsquared:.4f}')
print(f'R² ajustado:   {modelo.rsquared_adj:.4f}')
print(f'AIC:           {modelo.aic:.2f}')

---
## PARTE 3 — Verificación de supuestos

### 3.1 Supuesto 1 — Linealidad (scatter predictores vs Y)

In [ ]:
predictores = ['tiempo_estudio', 'interacciones_foro', 'sesiones_activas', 'notas_previas']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Supuesto de LINEALIDAD — cada predictor vs Y', fontweight='bold')

for ax, col in zip(axes, predictores):
    r = df[col].corr(Y)
    sns.regplot(data=df, x=col, y='rendimiento_final', ax=ax,
                ci=None, scatter_kws={'alpha': 0.4, 's': 20},
                line_kws={'color': '#ED7D31', 'linewidth': 2})
    ax.set_title(f'{col}\nr = {r:.3f}', fontsize=9)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

### 3.2 Supuesto 2 y 3 — Residuos vs valores ajustados (Independencia + Homocedasticidad)

In [ ]:
# Código exacto de la presentación
residuos     = modelo.resid
predicciones = modelo.fittedvalues

# Gráfico exacto de la presentación
sns.scatterplot(x=predicciones, y=residuos)
plt.axhline(0, color='red')
plt.title('Residuos vs. Predicción')
plt.xlabel('Valores ajustados')
plt.ylabel('Residuos')
plt.show()

In [ ]:
# Versión extendida con diagnóstico
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Homocedasticidad e Independencia — Residuos', fontweight='bold')

axes[0].scatter(predicciones, residuos, color='#2E75B6', alpha=0.6, s=30)
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
# Añadir banda ±2σ para referencia
s = residuos.std()
axes[0].axhline( 2*s, color='orange', linestyle=':', linewidth=1.5, label='+2σ')
axes[0].axhline(-2*s, color='orange', linestyle=':', linewidth=1.5, label='-2σ')
axes[0].set_title('Residuos vs Valores ajustados\n(los puntos deben estar distribuidos aleatoriamente)')
axes[0].set_xlabel('ŷ')
axes[0].set_ylabel('Residuo')
axes[0].legend(fontsize=8)

# Residuos vs orden (independencia)
axes[1].plot(range(len(residuos)), residuos, color='#70AD47', alpha=0.7, linewidth=1)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuos vs Índice\n(no debe haber patrón temporal)')
axes[1].set_xlabel('Índice de observación')
axes[1].set_ylabel('Residuo')

plt.tight_layout()
plt.show()

n_outliers = (np.abs(residuos) > 2*s).sum()
print(f'Residuos fuera de ±2σ: {n_outliers} ({n_outliers/len(residuos)*100:.1f}%)')
print('Esperado en dist. normal: ~5%')

### 3.3 Supuesto 4 — Normalidad de los residuos (Q-Q plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Supuesto de NORMALIDAD de residuos', fontweight='bold')

# Q-Q plot exacto de la presentación
sm.qqplot(residuos, line='45', ax=axes[0])
axes[0].set_title('Gráfico Q-Q de residuos\n(puntos deben seguir la línea roja)')

# Histograma de residuos
sns.histplot(residuos, bins=15, kde=True, ax=axes[1], color='#BDD7EE', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Histograma de residuos\n(debe ser ~campana centrada en 0)')
axes[1].set_xlabel('Residuo')

plt.tight_layout()
plt.show()

# Test de Shapiro-Wilk
stat, p_shapiro = stats.shapiro(residuos)
print(f'Test Shapiro-Wilk: W={stat:.4f}, p={p_shapiro:.4f}')
print(f'Normalidad: {"✅ No se rechaza (p>0.05)" if p_shapiro > 0.05 else "⚠️ Se rechaza normalidad (p<0.05)"}')

### 3.4 Supuesto 5 — No multicolinealidad (VIF)

In [ ]:
# VIF — recomendado por la presentación
X_vif = df[predictores].copy()
X_vif = sm.add_constant(X_vif)

vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])]
}).set_index('Variable').round(2)

print('=== Variance Inflation Factor (VIF) ===')
print(vif_data)
print()
print('Referencia:')
print('  VIF < 5     → sin problemas de multicolinealidad ✅')
print('  VIF 5–10    → multicolinealidad moderada ⚠️')
print('  VIF > 10    → multicolinealidad severa ❌ — eliminar o combinar la variable')

# Visualización VIF
vif_plot = vif_data.drop('const').reset_index()
colores_vif = ['#FC4E4E' if v > 10 else '#FFC000' if v > 5 else '#70AD47'
               for v in vif_plot['VIF']]
plt.figure(figsize=(8, 3))
plt.barh(vif_plot['Variable'], vif_plot['VIF'], color=colores_vif, edgecolor='white')
plt.axvline(5,  color='orange', linestyle='--', linewidth=2, label='Umbral moderado (5)')
plt.axvline(10, color='red',    linestyle='--', linewidth=2, label='Umbral severo (10)')
plt.title('VIF por predictor', fontweight='bold')
plt.xlabel('VIF')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

### ✏️ Ejercicio 3 — Evaluación de supuestos:

In [ ]:
# ✏️ ¿Se cumple la linealidad? ¿Algún predictor muestra patrón no lineal?
r_linealidad = ""

# ✏️ ¿El gráfico de residuos muestra patrón sistemático? ¿Hay homocedasticidad?
r_homoced = ""

# ✏️ ¿Los residuos parecen normales según el Q-Q y el histograma?
r_normalidad = ""

# ✏️ ¿Hay multicolinealidad problemática? ¿Qué variables la presentan?
r_vif = ""

print('Linealidad:      ', r_linealidad)
print('Homocedasticidad:', r_homoced)
print('Normalidad:      ', r_normalidad)
print('Multicolinealidad:', r_vif)

---
## PARTE 4 — Selección de variables

### 4.1 Modelo reducido — eliminar variable no significativa (Backward)

In [ ]:
# Código exacto de la presentación
X_reducido = df[['tiempo_estudio', 'interacciones_foro', 'notas_previas']]
X_reducido = sm.add_constant(X_reducido)
modelo_reducido = sm.OLS(Y, X_reducido).fit()
print(modelo_reducido.summary())

### 4.2 Comparación sistemática de modelos

In [ ]:
# Comparar tres modelos: completo, reducido, mínimo
X_min = sm.add_constant(df[['tiempo_estudio', 'notas_previas']])
modelo_min = sm.OLS(Y, X_min).fit()

comparacion = pd.DataFrame({
    'Modelo': ['Completo (4 vars)', 'Reducido (3 vars)', 'Mínimo (2 vars)'],
    'Variables': [
        'tiempo, foro, sesiones, notas_prev',
        'tiempo, foro, notas_prev',
        'tiempo, notas_prev'
    ],
    'R²':       [modelo.rsquared,         modelo_reducido.rsquared,         modelo_min.rsquared],
    'R² adj':   [modelo.rsquared_adj,      modelo_reducido.rsquared_adj,     modelo_min.rsquared_adj],
    'AIC':      [modelo.aic,              modelo_reducido.aic,              modelo_min.aic],
    'BIC':      [modelo.bic,              modelo_reducido.bic,              modelo_min.bic],
    'N vars':   [4, 3, 2]
})
comparacion[['R²','R² adj']] = comparacion[['R²','R² adj']].round(4)
comparacion[['AIC','BIC']]   = comparacion[['AIC','BIC']].round(2)

print('=== Comparación de modelos ===')
print(comparacion.to_string(index=False))
print()
print('Regla: menor AIC/BIC = mejor equilibrio ajuste-complejidad')
mejor_aic = comparacion.loc[comparacion['AIC'].idxmin(), 'Modelo']
print(f'Mejor modelo por AIC: {mejor_aic}')

In [ ]:
# Visualizar comparación
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Comparación de modelos', fontweight='bold')

nombres = ['Completo\n(4 vars)', 'Reducido\n(3 vars)', 'Mínimo\n(2 vars)']
colores = ['#2E75B6', '#70AD47', '#ED7D31']

for ax, met, vals in zip(
    axes,
    ['R² ajustado', 'AIC', 'BIC'],
    [[m.rsquared_adj for m in [modelo, modelo_reducido, modelo_min]],
     [m.aic          for m in [modelo, modelo_reducido, modelo_min]],
     [m.bic          for m in [modelo, modelo_reducido, modelo_min]]]
):
    bars = ax.bar(nombres, vals, color=colores, edgecolor='white')
    ax.set_title(met, fontweight='bold')
    if 'R²' in met:
        ax.set_ylim(0, 1)
        ax.set_ylabel('R² ajustado (mayor = mejor)')
    else:
        ax.set_ylabel(f'{met} (menor = mejor)')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v,
                f'{v:.3f}' if 'R²' in met else f'{v:.1f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### 4.3 Backward elimination automatizado (por p-value)

In [ ]:
def backward_elimination(X_df, Y, threshold=0.05, verbose=True):
    """Elimina iterativamente la variable con mayor p-value si supera el umbral."""
    variables = list(X_df.columns)
    paso = 0
    while True:
        X_sm = sm.add_constant(X_df[variables])
        modelo_iter = sm.OLS(Y, X_sm).fit()
        pvalues = modelo_iter.pvalues.drop('const')
        max_pval = pvalues.max()
        if max_pval > threshold:
            eliminar = pvalues.idxmax()
            paso += 1
            if verbose:
                print(f'Paso {paso}: eliminar "{eliminar}" (p={max_pval:.4f} > {threshold})')
            variables.remove(eliminar)
        else:
            break
    if verbose:
        print(f'\nVariables finales: {variables}')
        print(f'R² ajustado: {modelo_iter.rsquared_adj:.4f}')
        print(f'AIC:         {modelo_iter.aic:.2f}')
    return variables, modelo_iter

print('=== Backward Elimination (umbral p < 0.05) ===')
vars_finales, modelo_backward = backward_elimination(
    df[['tiempo_estudio','interacciones_foro','sesiones_activas','notas_previas']],
    Y
)

### ✏️ Ejercicio 4 — Selección:

In [ ]:
# ✏️ ¿Qué variable eliminó backward elimination? ¿Qué indica eso sobre su aporte al modelo?
r_elim = ""

# ✏️ ¿Qué diferencia hay entre usar R² y AIC para seleccionar el mejor modelo?
r_r2_vs_aic = ""

# ✏️ ¿Cuándo podrías querer conservar una variable aunque su p-value sea > 0.05?
r_conservar = ""

print(f'Variable eliminada:  {r_elim}')
print(f'R² vs AIC:           {r_r2_vs_aic}')
print(f'Conservar con p>0.05: {r_conservar}')

---
## PARTE 5 — Actividad autónoma: Rendimiento académico

### 5.1 Cargar y diagnosticar

In [ ]:
# Código exacto de la presentación
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

df2 = pd.read_csv('datos_regresion_multiple_autonomo.csv')
print(df2.describe().round(3))

In [ ]:
print('=== Valores fuera de rango ===')
print(df2[df2['nota_diagnostico'] > 7][['id_estudiante','nota_diagnostico','nota_final']])
print()
# Limpiar el outlier intencional
df2_clean = df2[df2['nota_diagnostico'] <= 7].copy()
print(f'Registros: {len(df2)} → {len(df2_clean)} después de limpieza')

### 5.2 Ajustar modelo completo

In [ ]:
# Código exacto de la presentación
X2 = df2_clean[['horas_estudio', 'participaciones_chat', 'videos_vistos', 'nota_diagnostico']]
X2 = sm.add_constant(X2)
y2 = df2_clean['nota_final']
modelo2 = sm.OLS(y2, X2).fit()
print(modelo2.summary())

### 5.3 Verificar supuestos

In [ ]:
res2  = modelo2.resid
pred2 = modelo2.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Verificación de supuestos — Modelo autónomo', fontweight='bold', fontsize=12)

# 1. Residuos vs ajustados
axes[0,0].scatter(pred2, res2, color='#2E75B6', alpha=0.6, s=30)
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0,0].set_title('Residuos vs Valores ajustados')
axes[0,0].set_xlabel('ŷ')
axes[0,0].set_ylabel('Residuo')

# 2. Q-Q plot
sm.qqplot(res2, line='45', ax=axes[0,1])
axes[0,1].set_title('Q-Q plot — Normalidad de residuos')

# 3. Histograma residuos
sns.histplot(res2, bins=15, kde=True, ax=axes[1,0], color='#BDD7EE', edgecolor='white')
axes[1,0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1,0].set_title('Histograma de residuos')

# 4. VIF
X2_vif = sm.add_constant(df2_clean[['horas_estudio','participaciones_chat',
                                     'videos_vistos','nota_diagnostico']])
vif2 = pd.DataFrame({
    'Variable': X2_vif.columns,
    'VIF': [variance_inflation_factor(X2_vif.values, i)
            for i in range(X2_vif.shape[1])]
}).set_index('Variable').drop('const').reset_index()

colores_v = ['#FC4E4E' if v > 10 else '#FFC000' if v > 5 else '#70AD47'
             for v in vif2['VIF']]
axes[1,1].barh(vif2['Variable'], vif2['VIF'], color=colores_v, edgecolor='white')
axes[1,1].axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='Umbral 5')
axes[1,1].axvline(10, color='red',    linestyle='--', linewidth=1.5, label='Umbral 10')
axes[1,1].set_title('VIF por predictor')
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.show()

stat2, p_shap2 = stats.shapiro(res2)
print(f'Shapiro-Wilk: p={p_shap2:.4f} → {"✅ Normalidad OK" if p_shap2 > 0.05 else "⚠️ Revisar normalidad"}')

### 5.4 Modelo reducido y comparación

In [ ]:
# Código exacto de la presentación
X2_reducido = df2_clean[['horas_estudio', 'nota_diagnostico', 'videos_vistos']]
X2_reducido = sm.add_constant(X2_reducido)
modelo2_reducido = sm.OLS(y2, X2_reducido).fit()
print(modelo2_reducido.summary())

### ✏️ Conclusiones actividad autónoma:

In [ ]:
print('=== Comparación modelos autónomo ===')
comp2 = pd.DataFrame({
    'Modelo':   ['Completo (4 vars)', 'Reducido (3 vars)'],
    'R²':       [modelo2.rsquared, modelo2_reducido.rsquared],
    'R² adj':   [modelo2.rsquared_adj, modelo2_reducido.rsquared_adj],
    'AIC':      [modelo2.aic, modelo2_reducido.aic],
}).round(4)
print(comp2.to_string(index=False))
print()

# ✏️ Interpreta los coeficientes del modelo reducido:
c1 = ""
# ✏️ ¿Qué variables descartaste y por qué?
c2 = ""
# ✏️ ¿Se cumplen los supuestos? ¿Cuál presenta más dudas?
c3 = ""
# ✏️ ¿Cómo usarías este modelo en un contexto profesional real?
c4 = ""

print('--- CONCLUSIONES ---')
for i, c in enumerate([c1, c2, c3, c4], 1):
    print(f'{i}. {c}')

---
## 📋 Resumen: supuestos, diagnósticos y selección

| Supuesto | Cómo verificarlo | Señal de problema |
|----------|-----------------|-------------------|
| **Linealidad** | Scatter X vs Y | Patrón curvilíneo |
| **Independencia** | Residuos vs índice | Patrón temporal/serial |
| **Homocedasticidad** | Residuos vs ŷ | Embudo o patrón creciente |
| **Normalidad** | Q-Q plot + Shapiro | Puntos fuera de la diagonal |
| **No multicolinealidad** | VIF | VIF > 5 (moderado), > 10 (severo) |

**Selección de variables:**

| Método | Descripción | Código |
|--------|-------------|--------|
| Backward | Eliminar variable con mayor p-value iterativamente | Manual con bucle |
| AIC | Menor AIC = mejor ajuste penalizado | `modelo.aic` |
| BIC | Penaliza más fuertemente que AIC | `modelo.bic` |
| R² ajustado | Penaliza variables innecesarias | `modelo.rsquared_adj` |
| VIF | Detectar multicolinealidad | `variance_inflation_factor()` |

> 💡 **Ceteris paribus:** Cada coeficiente β₁ se interpreta como el efecto de X₁ sobre Y **manteniendo todas las demás variables constantes**. Esto es lo que distingue la regresión múltiple del análisis de correlaciones simples.

> 💡 **No solo R²:** Un modelo con R²=0.95 pero con VIF>10 o residuos con patrón sistemático es un modelo con problemas. Siempre verifica los supuestos.